# Speculative decoding on a Kaggle T4

Runs the [mini_project](https://github.com/nithinj25/mini_project) experiments on Kaggle's free T4. A T4 is the "T4-class GPU" the SPEC's claim is about.

**Before running:**
1. **Settings → Accelerator → GPU T4** (a single T4, or 2× T4; only one is used). Avoid the P100, which recent PyTorch builds may not support.
2. **Settings → Internet → On.** This needs a phone-verified Kaggle account.
3. For long plans, use **Save Version → Save & Run All (Commit)**. It runs in the background, up to Kaggle's session limit, and keeps everything in `/kaggle/working` as notebook output.

**Plans** (rough GPU time):

| Plan | Steps | Time |
| --- | --- | --- |
| `QUICK` | tests, Phase 1 (c and baseline), Phase 5b (O2 with the CUDA-graph draft) | ~1.5–2 h |
| `CORE` | QUICK + Phase 2, Phase 4, Phase 5 (O2 with the eager draft) | ~3–3.5 h |
| `FULL` | CORE + Phase 3 (the fixed-γ sweep), results.csv and all figures | ~6 h |

Results go to `results/<GPU>/` inside the repo, so they never mix with the laptop's numbers, and are zipped to `/kaggle/working/specdec_results_<GPU>.zip`. Every step is resumable: if a session dies, run the notebook again and finished work is skipped.

In [ ]:
PLAN = "QUICK"          # QUICK | CORE | FULL
REPO = "https://github.com/nithinj25/mini_project.git"   # must be public, or see the note at the end

In [ ]:
# 1. GPU check
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
                     capture_output=True, text=True).stdout)
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout
assert gpu.strip(), "No GPU: Settings -> Accelerator -> GPU T4"
if "T4" not in gpu:
    print("WARNING: not a T4. The P100 may be unsupported by the installed PyTorch; results are still labelled by GPU.")

In [ ]:
# 2. Get the code (clone, or update an existing copy)
import os
os.chdir("/kaggle/working")
if os.path.isdir("mini_project/.git"):
    print(subprocess.run(["git", "-C", "mini_project", "pull", "--ff-only"], capture_output=True, text=True).stdout)
else:
    r = subprocess.run(["git", "clone", "--depth", "1", REPO, "mini_project"], capture_output=True, text=True)
    print(r.stdout, r.stderr)
    assert r.returncode == 0, "clone failed: is Internet on, and is the repo public?"
os.chdir("/kaggle/working/mini_project")
print(subprocess.run(["git", "log", "-1", "--format=%h %ci %s"], capture_output=True, text=True).stdout)

In [ ]:
# 3. Dependencies. Kaggle's preinstalled PyTorch is kept; transformers must be >= 4.56.
!pip install -q -U "transformers>=4.56" nvidia-ml-py accelerate
import importlib.metadata as md
print("torch", md.version("torch"), "| transformers", md.version("transformers"))

In [ ]:
# 4. Run the plan
import os, re
gpu_tag = re.sub(r"[^A-Za-z0-9]+", "_", gpu.strip().splitlines()[0].replace("Tesla", "").replace("NVIDIA", "")).strip("_")
os.environ["SPECDEC_RESULTS"] = f"results/{gpu_tag}"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # 2x T4: use one card; both models must share a device
steps = {"QUICK": "tests,p1,p5b",
         "CORE":  "tests,p1,p2,p4,p5,p5b,results,plots",
         "FULL":  "tests,p1,p2,p3,p4,p5,p5b,results,plots"}[PLAN]
print("results dir:", os.environ["SPECDEC_RESULTS"], "| steps:", steps)
!python run_all.py --steps {steps} --zip /kaggle/working/specdec_results_{gpu_tag}.zip

In [ ]:
# 5. Headline numbers
import json, os
d = os.environ["SPECDEC_RESULTS"]
def show(name, keys):
    p = os.path.join(d, name)
    if os.path.exists(p):
        j = json.load(open(p))
        print(f"{name}:", {k: j[k] for k in keys if k in j})
show("phase1.json", ["gpu", "dtype", "c_mean", "c_std", "baseline_tokens_per_s_mean", "predicted_best_gamma"])
show("phase2.json", ["matches", "runs", "alpha_by_task"])
for name in ("phase5_summary.json", "phase5b_summary.json"):
    p = os.path.join(d, name)
    if os.path.exists(p):
        j = json.load(open(p))
        print(f"\n{name}: cells {j['cells']}, c per cell {j['c_per_cell']}")
        for opp, comp in j["comparisons"].items():
            a = comp["all"]
            print(f"  adaptive vs {opp}: d={a['mean_diff']:+.3f}  95% CI {a['ci95']}  wins {a['wins']}/{a['n']}  -> {a['verdict']}")

## Getting the results back

- **Download:** the notebook's **Output** tab (after *Save & Run All*) has `specdec_results_<GPU>.zip`. Unzip it into the repo as `results/<GPU>/`, next to the laptop's `results/`.
- Every file inside has the same format as the laptop results, so `make_results.py` and `plots.py` work on it directly with `SPECDEC_RESULTS=results/<GPU>`.
- `results/<GPU>/logs/environment.json` records the GPU, PyTorch and transformers versions used.

**Private repo?** Add a GitHub token as a Kaggle Secret named `GITHUB_TOKEN` (Add-ons → Secrets). Then set `REPO = f"https://{token}@github.com/nithinj25/mini_project.git"` with `token = UserSecretsClient().get_secret("GITHUB_TOKEN")`, imported from `kaggle_secrets`.